# SPL3: Retrain FraudGNN with the injected fraud patterns

**আগে যা করবেন:** Google Drive-এর `MyDrive/SPL3/` ফোল্ডারে এই ৬টা ফাইল upload করুন (পুরনো থাকলে replace করুন):
`00_inject_fraud_patterns.py`, `01_load_paysim.py`, `03_graph_builder.py`, `main_gnn_model.py`, `train_main_gnn.py`, `09_evaluate_patterns.py`

আসল PaySim CSV (`PS_20174392719_1491204439457_log.csv`) আগে থেকেই সেখানে আছে, নতুন করে upload করতে হবে না।

**Runtime → Change runtime type → T4 GPU** বেছে নিন। তারপর cell-গুলো উপর থেকে নিচে একটা একটা করে চালান।

## ১. Drive mount আর SPL3 ফোল্ডারে যাওয়া

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SPL3

## ২. Library install (scikit-learn backend-এর সাথে একই version: 1.9.0)

এই cell চালানোর পর যদি লেখা আসে **RESTART NEEDED**, তাহলে **Runtime → Restart session** করুন, তারপর **cell ১ আবার চালিয়ে** সরাসরি cell ৩-এ চলে যান। Cell ২ আর চালাতে হবে না।

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn==1.9.0
import importlib.metadata as md
print("scikit-learn installed:", md.version("scikit-learn"))
import sklearn
print("scikit-learn loaded:   ", sklearn.__version__)
print("OK" if sklearn.__version__ == "1.9.0" else "RESTART NEEDED: Runtime -> Restart session, then run cell 1 and continue from cell 3")

## ৩. সব ফাইল আর GPU ঠিক আছে কিনা দেখা

In [ ]:
%cd /content/drive/MyDrive/SPL3
import os, sklearn, torch
need = ["PS_20174392719_1491204439457_log.csv", "00_inject_fraud_patterns.py", "01_load_paysim.py",
        "03_graph_builder.py", "main_gnn_model.py", "train_main_gnn.py", "09_evaluate_patterns.py"]
for f in need:
    print(("OK      " if os.path.exists(f) else "MISSING ") + f)
print("previous model:", "found (will be kept as backup)" if os.path.exists("models/FraudGNN_main.pt") else "not found (no before/after comparison)")
print("scikit-learn", sklearn.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

## ৪. Fraud pattern inject করা (~২-৫ মিনিট)
আসল CSV বদলায় না। নতুন `PS_augmented.csv` আর `injected_manifest.csv` তৈরি হয়।

In [ ]:
%cd /content/drive/MyDrive/SPL3
!python 00_inject_fraud_patterns.py --src PS_20174392719_1491204439457_log.csv --out-dir .

## ৫. Data clean + scale (~৩-৫ মিনিট)
`Tagged 15,636 injected rows` লেখা দেখতে পাবেন। পুরনো scaler `scaler_v1_paysim_only.pkl` নামে রেখে দেয়।

In [ ]:
%cd /content/drive/MyDrive/SPL3
!python 01_load_paysim.py

## ৬. Graph তৈরি (~১০-২০ মিনিট)
`Holdout (injected, test-only) nodes: 2888` দেখতে পাবেন।

In [ ]:
%cd /content/drive/MyDrive/SPL3
!python 03_graph_builder.py

## ৭. Training (সবচেয়ে বেশি সময়, প্রায় ৩০-৯০ মিনিট)

প্রতি ৩ epoch-এ একটা লাইন আসবে। প্রথম ৩০ epoch "warm-up", তারপর `patience` গোনা শুরু। এই সময়ে report বা slide-এর কাজ করতে পারেন। **শুধু browser tab বন্ধ করবেন না।**

শেষে পুরনো model `models/FraudGNN_main_v1_paysim_only.pt` নামে থেকে যাবে, আর নতুনটা `models/FraudGNN_main.pt`।

In [ ]:
%cd /content/drive/MyDrive/SPL3
!python train_main_gnn.py

## ৮. Model pattern শিখেছে কিনা পরীক্ষা (~৫-১৫ মিনিট)

শেষে **Checklist A-E** আসবে, প্রতিটার পাশে PASS/FAIL। পুরো output-টা copy করে আমাকে পাঠান।

In [ ]:
%cd /content/drive/MyDrive/SPL3
!python 09_evaluate_patterns.py

## ৯. Report-এর chart দেখা

In [ ]:
%cd /content/drive/MyDrive/SPL3
from IPython.display import Image, display
display(Image("pattern_recall.png"))

## ১০. Laptop-এ যা download করবেন

| Drive-এ ফাইল | Laptop-এ কোথায় রাখবেন |
|---|---|
| `models/FraudGNN_main.pt` | `D:/SPL3/backend/model/FraudGNN_main.pt` (replace) |
| `scaler.pkl` | `D:/SPL3/backend/model/scaler.pkl` (replace) |
| `type_encoder.pkl` | `D:/SPL3/backend/model/type_encoder.pkl` (replace) |
| `pattern_eval_results.csv`, `neighbour_effect.csv`, `graph_vs_xgb_separation.csv`, `pattern_recall.png`, `main_gnn_results.csv` | report-এর জন্য যেকোনো ফোল্ডারে |

⚠️ **model, scaler আর type_encoder তিনটা একসাথে replace করবেন।** একটা পুরনো আর একটা নতুন থাকলে score ভুল আসবে।